# E4 — Normal-aware loss screening (UNet 224×224)

Before saving this notebook version: attach the Kaggle Dataset containing the preprocessed BTXRD structure and enable Internet so the exact Git branch can be cloned. This notebook runs six losses × 30 epochs and never evaluates the test split.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import torch
import yaml

REPO_URL = 'https://github.com/lehngoc/BTXRD-LViT.git'
BRANCH = 'model/loss-ablation-normal-fp'
REPO_ROOT = Path('/kaggle/working/BTXRD-LViT')

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle Notebook Settings.'
print('GPU:', torch.cuda.get_device_name(0))

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'PyYAML'], check=True)
print(subprocess.check_output(['git', '-C', str(REPO_ROOT), 'log', '-1', '--oneline'], text=True))

In [ ]:
def find_data_root() -> Path:
    suffix = 'data/exports/btxrd_preprocessed/train.csv'
    for csv_path in Path('/kaggle/input').rglob('train.csv'):
        if csv_path.as_posix().endswith(suffix):
            return csv_path.parents[3]
    raise FileNotFoundError('Attach the preprocessed BTXRD Kaggle Dataset before running.')

DATA_ROOT = find_data_root()
required = [
    DATA_ROOT / 'data/exports/btxrd_preprocessed/train.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/val.csv',
    DATA_ROOT / 'data/exports/btxrd_preprocessed/test.csv',
    DATA_ROOT / 'data/processed/images_preprocessed',
    DATA_ROOT / 'data/processed/masks_preprocessed',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, '\n'.join(missing)
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
RUNTIME_DIR = Path('/kaggle/working/runtime_configs/screening')
EXPERIMENT_ROOT = Path('/kaggle/working/experiments/loss_ablation/screening')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

def make_runtime_config(source: Path, loss_id: str) -> Path:
    cfg = yaml.safe_load(source.read_text(encoding='utf-8'))
    cfg['data']['root_dir'] = str(DATA_ROOT)
    cfg['training']['device'] = 'cuda'
    cfg['training']['num_workers'] = 2
    cfg['training']['epochs'] = 30
    cfg['training']['seed'] = 42
    cfg['training']['output_dir'] = str(EXPERIMENT_ROOT / loss_id / 'seed42')
    destination = RUNTIME_DIR / f'{loss_id}.yaml'
    destination.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    return destination

source_configs = sorted((REPO_ROOT / 'configs/loss_ablation').glob('*.yaml'))
assert len(source_configs) == 6, source_configs

for source in source_configs:
    loss_id = yaml.safe_load(source.read_text(encoding='utf-8'))['experiment']['loss_id']
    runtime_config = make_runtime_config(source, loss_id)
    print(f'===== screening: {loss_id} =====')
    subprocess.run([sys.executable, '-m', 'src.training.train_unet', '--config', str(runtime_config)], cwd=REPO_ROOT, check=True)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.training.aggregate_loss_ablation',
    '--runs-root', str(EXPERIMENT_ROOT), '--stage', 'screening',
], cwd=REPO_ROOT, check=True)

summary_path = EXPERIMENT_ROOT / 'screening_loss_ablation_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps({
    'best_challenger': summary['best_challenger'],
    'validation_winner_overall': summary['validation_winner_overall'],
    'ranked_challengers': summary['ranked_challengers'],
}, indent=2))

archive = shutil.make_archive('/kaggle/working/loss_ablation_screening_artifacts', 'gztar',
                              root_dir='/kaggle/working', base_dir='experiments/loss_ablation/screening')
print('Saved archive:', archive)